# 04 - DPR: Dense Passage Retrieval

This notebook walks you through:

1. Installing the extra dependencies needed for DPR
2. Indexing passages (encoding + storing dense vectors in PostgreSQL)
3. Running search queries with DPR
4. Evaluating retrieval quality against the `qrels` ground truth

 > **Prerequisite:** You must have run `01_data_preparation.ipynb` first so the
 > `passages`, `queries`, and `qrels` tables are populated.

In [2]:
import sys
from pathlib import Path

# Resolve project root from notebook location
project_root = Path.cwd().resolve().parent
print(f"Project root: {project_root}")

# Ensure project root is importable so `src` package can be resolved from notebooks/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Check for .env file at project root
env_path = project_root / ".env"
if env_path.exists():
    print(f"Found .env file: {env_path}")
else:
    print("No .env found at project root. Default values from config.py will be used.")
    print("You can create one manually if needed.")

Project root: /home/tim/Documents/Projet_Big_Data_IR
Found .env file: /home/tim/Documents/Projet_Big_Data_IR/.env


## 1) Install Python dependencies

Run this once per environment.

In [3]:
import sys
import subprocess

requirements_file = project_root / "requirements.txt"
print(f"Installing dependencies from: {requirements_file}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements_file)],
    check=True,
)

Installing dependencies from: /home/tim/Documents/Projet_Big_Data_IR/requirements.txt


CompletedProcess(args=['/home/tim/Documents/Projet_Big_Data_IR/.venv/bin/python', '-m', 'pip', 'install', '-r', '/home/tim/Documents/Projet_Big_Data_IR/requirements.txt'], returncode=0)

## 2) Index passages with DPR

This step encodes passages from the `passages` table and stores the resulting dense vectors in the `dpr` table.

You can re-run safely - already-indexed passages are skipped.

In [7]:
from src.dpr.indexer import index_passages

# Use None to index all remaining passages once; reruns will skip automatically.
index_passages(batch_size=128)

2026-03-17 19:56:07,200 - INFO - Database connection established successfully.


Passages missing in dpr: 656193
Indexing up to 656193 missing passages...
Encoding 656193 passages (this may take a few minutes)...


Batches:   0%|          | 0/5127 [00:15<?, ?it/s]


KeyboardInterrupt: 

### Verify indexing

In [4]:
import pandas as pd
import warnings
from src.database.connection import get_connection

warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy.*")

conn = get_connection()
try:
    count = pd.read_sql_query("SELECT COUNT(*) AS indexed FROM dpr", conn)
    print(f"Indexed passages: {count['indexed'][0]}")

    print("\nSample DPR entries:")
    sample = pd.read_sql_query(
        "SELECT passage_id FROM dpr ORDER BY passage_id LIMIT 5",
        conn
    )
    display(sample)
finally:
    conn.close()

2026-03-17 19:48:55,562 - INFO - Database connection established successfully.


Indexed passages: 20000

Sample DPR entries:


,passage_id
0,1
1,2
2,3
3,4
4,5


## 3) Search with DPR

DPR performs dense retrieval by encoding the query and ranking passages with pgvector distance.

### Run a search query

In [5]:
from src.dpr.search import search_query

query = "what is the speed of light"

search_query(query, top_k=10)

/home/tim/Documents/Projet_Big_Data_IR/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-17 19:49:15,972 - INFO - Use pytorch device_name: cpu
2026-03-17 19:49:15,973 - INFO - Load pretrained SentenceTransformer: msmarco-distilbert-base-v4
2026-03-17 19:49:16,126 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/msmarco-distilbert-base-v4/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-17 19:49:16,142 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/msmarco-distilbert-base-v4/b2f66c95aba1481a880479165582020c2b9b64d7/modules.json "HTTP/1.1 200 OK"


Loading model msmarco-distilbert-base-v4 (768 dim)...


2026-03-17 19:49:16,252 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/msmarco-distilbert-base-v4/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-17 19:49:16,267 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/msmarco-distilbert-base-v4/b2f66c95aba1481a880479165582020c2b9b64d7/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-17 19:49:16,381 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/msmarco-distilbert-base-v4/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-17 19:49:16,400 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/msmarco-distilbert-base-v4/b2f66c95aba1481a880479165582020c2b9b64d7/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-17 19:49:16,512 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/msmarco-d

Model loaded successfully. Dimension: 768


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.14it/s]
2026-03-17 19:49:18,493 - INFO - Database connection established successfully.


Query: 'what is the speed of light'
Latency: 20.6 ms

--- TOP 10 RESULTS ---

#1 (passage_id=9200, score=0.5379)
Visible light has frequencies ranging from 4*10 14 Hz to 8*10 14 Hz (400 THz to 800 THz) and wavelengths from 3.8*10 -7 m to 7.5*10 -7 m (380 nm to 750 nm). Red light has the lowest frequency (longest wavelength) and violet light has the highest frequency (shortest wavelength) of visible light.

#2 (passage_id=9421, score=0.3028)
The Mountain of Light or Koh-i-Noor (Persian) is a diamond that was mined at Kollur Mine, in the present state of Andhra Pradesh in India. It was originally 793 carats when uncut. Once the largest known diamond, it is now a 105.6 metric carat diamond, weighing 21.6 grammes in its most recent cut state. In 1852, Albert the Prince Consort ordered it cut down from 186 carats. The diamond was originally owned by the Kakatiya dynasty, which had installed it in a temple of a Hindu goddess as her eye

#3 (passage_id=3295, score=0.2750)
Well, for one thing 

### Try your own queries

In [ ]:
from src.dpr.search import search_query

for q in ["how old is the earth", "who wrote hamlet", "symptoms of diabetes"]:
    print(f"\nRunning query: {q}")
    search_query(q, top_k=3)

2026-03-17 14:11:19,000 - INFO - Use pytorch device_name: cpu
2026-03-17 14:11:19,001 - INFO - Load pretrained SentenceTransformer: msmarco-distilbert-base-v4


Chargement du modèle msmarco-distilbert-base-v4 (768 dim)...


2026-03-17 14:11:19,224 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/msmarco-distilbert-base-v4/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-03-17 14:11:19,396 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/msmarco-distilbert-base-v4/b2f66c95aba1481a880479165582020c2b9b64d7/modules.json "HTTP/1.1 200 OK"
2026-03-17 14:11:19,553 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/sentence-transformers/msmarco-distilbert-base-v4/b2f66c95aba1481a880479165582020c2b9b64d7/modules.json "HTTP/1.1 200 OK"
2026-03-17 14:11:19,695 - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/msmarco-distilbert-base-v4/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-03-17 14:11:19,840 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/msmarco-distilbert-base-v4/b2f66c95aba1481a88047

Modèle chargé. Dimension : 768
Récupération de 10000 passages depuis la table 'passages'...
Encodage de 10000 passages en cours (cela peut prendre quelques minutes)...


Batches: 100%|██████████| 79/79 [11:18<00:00,  8.58s/it]


Insertion des vecteurs dans la base de données...
Terminé ! 10,000 embeddings de dimension 768 ont été insérés.


## 4) Evaluation - MRR@10

Compute **Mean Reciprocal Rank @ 10** using the ground-truth `qrels` table.

In [ ]:
from src.dpr.search import evaluate_mrr

# Increase eval_queries for a more stable estimate
evaluate_mrr(eval_queries=100, eval_top_k=10)

Batches: 100%|██████████| 1/1 [00:00<00:00, 40.55it/s]
2026-03-17 14:25:55,967 - INFO - Database connection established successfully.


Query: 'Who is Maddison?'
Latency: 8.0 ms

--- TOP 5 RESULTS ---

#1 (passage_id=1783, score=0.2282)
The film also stars Honor Blackman as Bond girl Pussy Galore and Gert Froebe fröbe as the title Character Auric, goldfinger along With Shirley eaton as the Iconic bond Girl Jill. Masterson goldfinger was produced By Albert. R broccoli And Harry saltzman and was the first of Four bond films directed By Guy. hamilton In 2008, Total Film named Goldfinger as the best film in the series. The Times placed Goldfinger and Oddjob second and third on their list of the best Bond villains in 2008. They also named the Aston Martin DB5 as the best car in the films.

#2 (passage_id=1785, score=0.2046)
It is based on the novel of the same name by Ian Fleming. The film also stars Honor Blackman as Bond girl Pussy Galore and Gert Froebe fröbe as the title Character Auric, goldfinger along With Shirley eaton as the Iconic bond Girl Jill. masterson In 2008, Total Film named Goldfinger as the best film in t

## 5) Architecture Summary

| Component | Detail |
|---|---|
| **Backbone** | `msmarco-distilbert-base-v4` |
| **Embedding type** | Dense passage vectors |
| **Similarity** | pgvector distance (`<=>`) converted to score |
| **Storage** | PostgreSQL `vector(768)` in `dpr` |
| **Indexing logic** | Insert only missing `passage_id` values |
| **Evaluation** | MRR@10 against `qrels` |

## Notes

- The first run downloads DPR model weights from Hugging Face.
- Indexing is idempotent: already-indexed passages are skipped.
- Search calls in this notebook use `src.dpr.search` and automatically log runs to `search_logs` and `results`.
- For large collections, tune PostgreSQL/pgvector indexing strategy to improve latency.